<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/12-caching-memory/00-storage-levels.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 12.0 — Storage levels: catch a DataFrame cache in the act

Cache a DataFrame, confirm it's columnar (not pickled Python objects),
watch `MEMORY_ONLY` silently recompute a partition that doesn't fit,
fix it with `MEMORY_AND_DISK`, and compare `cache()` / `persist()` /
`CACHE TABLE` as three doors into the same mechanism.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark import StorageLevel

spark = (
    SparkSession.builder
    .appName("12.0-storage-levels")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")  # stable teaching plans (Module 7/10 convention)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
# local[*] note: one JVM plays driver + all executors, so the Executors tab
# shows a single "driver" entry — the memory MODEL below is identical to a
# real cluster, only the topology is collapsed.

## `cache()` is lazy — materializes on the next action (2.4's rule, restated)

In [ ]:
df = spark.range(2_000_000).withColumn("k", (col("id") % 50).cast("int")) \
                             .withColumn("v", col("id") * 2)

df.cache()
print("isCached before any action:", spark.catalog.isCached  # not a temp view yet
      if False else "n/a (no view registered)")
df.count()   # forces materialization
print("storageLevel:", df.storageLevel)   # MEMORY_AND_DISK — DataFrame default, not MEMORY_ONLY

## Storage tab via the REST API (10.2's helper, new endpoint)

`.../storage/rdd` lists every cached "RDD" (a cached DataFrame is one
under the hood) with memory/disk bytes and fraction cached.

In [ ]:
import urllib.request, json as _json

def storage_report():
    app_id = spark.sparkContext.applicationId
    url = f"http://localhost:4040/api/v1/applications/{app_id}/storage/rdd"
    for r in _json.load(urllib.request.urlopen(url)):
        print(f"{r['name']:<20} fraction_cached={r['numCachedPartitions']}/{r['numPartitions']} "
              f"mem={r['memoryUsed']:,}B disk={r['diskUsed']:,}B")

storage_report()

## `MEMORY_ONLY` doesn't fit → silent per-partition recompute

Force a bigger dataset into `MEMORY_ONLY` deliberately, time two
actions. If a chunk didn't fit, the second action still pays some
recompute cost — compare against `MEMORY_AND_DISK` below.

In [ ]:
import time

big = spark.range(20_000_000).withColumn("k", (col("id") % 50).cast("int")) \
                              .withColumn("v", col("id") * 2) \
                              .withColumn("pad", F.lit("x" * 200))  # bulk up row size

big.persist(StorageLevel.MEMORY_ONLY)
t0 = time.time(); big.count(); print("first action (materializes):", round(time.time()-t0, 2), "s")
t0 = time.time(); big.count(); print("second action:               ", round(time.time()-t0, 2), "s")
storage_report()   # check fraction cached for 'big' — is it 100%?
big.unpersist()

## Fix: `MEMORY_AND_DISK` — overflow spills instead of recomputing

Same data, same size, different level. Compare the second-action
timing and the fraction cached.

In [ ]:
big.persist(StorageLevel.MEMORY_AND_DISK)
t0 = time.time(); big.count(); print("first action:", round(time.time()-t0, 2), "s")
t0 = time.time(); big.count(); print("second action:", round(time.time()-t0, 2), "s")
storage_report()
big.unpersist()

## Three doors, one mechanism: `cache()` / `persist()` / `CACHE TABLE`

In [ ]:
df.unpersist()
df.createOrReplaceTempView("t")

spark.sql("CACHE TABLE t")               # SQL door
print("catalog says cached:", spark.catalog.isCached("t"))
spark.sql("SELECT count(*) FROM t").collect()
storage_report()

spark.sql("UNCACHE TABLE t")
print("catalog says cached after uncache:", spark.catalog.isCached("t"))

## Anti-pattern: caching before a single write

Time a cache-then-write-once (wasted) against a plain write. The
cache buys nothing when there's exactly one downstream action.

In [ ]:
import shutil
shutil.rmtree("output", ignore_errors=True)

t0 = time.time()
df.cache()
df.write.mode("overwrite").parquet("output/wasted_cache")
print("cache + single write:", round(time.time()-t0, 2), "s")
df.unpersist()

t0 = time.time()
df.write.mode("overwrite").parquet("output/no_cache")
print("no cache,  single write:", round(time.time()-t0, 2), "s")
shutil.rmtree("output", ignore_errors=True)

In [ ]:
spark.stop()

## Exercises

1. Cache `df` at `DISK_ONLY` and compare timings against
   `MEMORY_ONLY` and `MEMORY_AND_DISK` for two back-to-back `count()`
   calls. Where does `DISK_ONLY` land, and why?
2. Build a query that filters a cached DataFrame
   (`df.cache(); df.count(); df.filter(col("v") > 1000).explain()`).
   Find `InMemoryTableScan` in the plan — does a filter pushed onto a
   cached table still show pruning info?
3. Cache `big` at `MEMORY_ONLY_2` (replicated). What does
   `storage_report()` show differently versus plain `MEMORY_ONLY`?
   (Local mode caveat: replication needs 2+ executors to be meaningful
   — note what you actually observe here.)
4. Reproduce the "cache before single write" anti-pattern, then fix it
   by removing the `cache()` call, and confirm via `storage_report()`
   that nothing is cached in the fixed version.
5. Use `spark.catalog.uncacheTable` vs `df.unpersist()` — cache the
   same data both ways (temp view + direct persist) and show both
   need to be released independently.

In [ ]:
# your exercise code here